# 02 – RL Agent Training (MaskablePPO + CNN)

Pełne podejście: `PacmanGridEnv` (obserwacja 2D `6×31×28` + maska legalnych akcji + reward shaping z PBRS),
algorytm **MaskablePPO** z małą siecią CNN, równoległe środowiska (`SubprocVecEnv`).

Naciśnij **⏹ Stop Training**, aby zatrzymać; checkpoint jest zapisywany co `CHECKPOINT_EVERY` kroków.

In [1]:
# Install sb3-contrib into THIS kernel (idempotent; %pip ensures correct interpreter).
import importlib, subprocess, sys
if importlib.util.find_spec("sb3_contrib") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sb3-contrib"])
    print("sb3-contrib installed — restart NOT required, just re-run the next cell.")
else:
    print("sb3-contrib already available.")

sb3-contrib already available.


In [2]:
import sys, threading, time, os
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym

import ipywidgets as widgets
from IPython.display import display

from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecMonitor
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

from src.environment.pacman_env import PacmanGridEnv
from src.utils.mlflow_logger import MLflowLogger

In [3]:
print(torch.cuda.is_available())

True


In [4]:
# ── Configuration ────────────────────────────────────────────────────────────
N_ENVS           = 8           # parallel environments
N_STEPS          = 256         # rollout length per env
TOTAL_TIMESTEPS  = 5_000_000   # upper bound (Stop button can interrupt earlier)
CHECKPOINT_PATH  = os.path.join('..', 'models', 'ppo_pacman')
CHECKPOINT_EVERY = 100_000
LOG_EVERY        = 10_000

# ── Action-mask wrapper (MaskablePPO requires this on each env) ──────────────
def mask_fn(env):
    return env.action_masks()

def make_env(seed: int):
    def _f():
        env = PacmanGridEnv(
            seed=seed,
            max_steps=5000,
            step_penalty=-0.01,
            reward_scale_div=100.0,
            pbrs_coef=0.2,
        )
        env = ActionMasker(env, mask_fn)
        return env
    return _f

# Build vectorised env
env_fns = [make_env(seed=i) for i in range(N_ENVS)]
vec_env = SubprocVecEnv(env_fns) if N_ENVS > 1 else DummyVecEnv(env_fns)
vec_env = VecMonitor(vec_env)

# ── Custom Global CNN suited for 6×31×28 input with Striders ─────────────────
class PacmanCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        c, h, w = observation_space.shape
        self.cnn = nn.Sequential(
            # Layer 1: 3x3 Conv, stride 1. Receptive field: 3x3
            nn.Conv2d(c, 32, kernel_size=3, padding=1), nn.ReLU(),
            # Layer 2: 3x3 Conv, stride 2. Downsamples 31x28 -> 16x14. Receptive field: 7x7
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            # Layer 3: 3x3 Conv, stride 2. Downsamples 16x14 -> 8x7. Receptive field: 15x15
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            # Layer 4: 3x3 Conv, stride 1. Receptive field: 23x23 (covers almost the entire board!)
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            n_flat = self.cnn(torch.zeros(1, c, h, w)).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flat, features_dim), nn.ReLU())

    def forward(self, x):
        return self.linear(self.cnn(x))

policy_kwargs = dict(
    features_extractor_class=PacmanCNN,
    features_extractor_kwargs=dict(features_dim=256),
    net_arch=dict(pi=[128, 128], vf=[128, 128]),
)

checkpoint_zip = CHECKPOINT_PATH + ".zip"
if os.path.exists(checkpoint_zip):
    print("[Auto-Resume] LOADING SUCCESSFUL")
    model = MaskablePPO.load(CHECKPOINT_PATH, env=vec_env)
else:
    print("[Initial Run] STARTING FROM SCRATCH")
    model = MaskablePPO(
        "CnnPolicy",
        vec_env,
        learning_rate=2.5e-4,
        n_steps=N_STEPS,
        batch_size=512,
        n_epochs=4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.1,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        verbose=0,
        device="auto",
    )

print(f"Model ready. Obs shape: {vec_env.observation_space.shape}, n_envs={N_ENVS}")
print(f"Device: {model.device}")

[Initial Run] STARTING FROM SCRATCH
Model ready. Obs shape: (6, 31, 28), n_envs=8
Device: cuda


# ── RL Agent Training (MaskablePPO + CNN) ───────────────────────────────────

We run the training loop in the **foreground**. This ensures all training outputs and error tracebacks are printed directly to the standard output and are 100% visible.

* **To Stop Training**: Simply click the native **Jupyter Stop/Interrupt** button at the top of this cell in VS Code at any time! The current model will be saved.

In [5]:
import time
import os
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback

class ConsoleCallback(BaseCallback):
    def __init__(self, checkpoint_path, checkpoint_every, log_every, mlflow_logger, total_steps):
        super().__init__()
        self.checkpoint_path  = checkpoint_path
        self.checkpoint_every = checkpoint_every
        self.log_every        = log_every
        self.mlflow_logger    = mlflow_logger
        self.total_steps      = total_steps
        self._next_log        = log_every
        self._next_ckpt       = checkpoint_every
        self._ep_rewards      = []
        self._ep_lengths      = []
        self._ep_scores       = []
        self._ep_levels       = []
        self._t0              = time.time()

    def _on_step(self) -> bool:
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self._ep_rewards.append(info['episode']['r'])
                self._ep_lengths.append(info['episode']['l'])
                self._ep_scores.append(info.get('score', 0))
                self._ep_levels.append(info.get('level', 1))

        n = self.num_timesteps
        if n >= self._next_log:
            elapsed = time.time() - self._t0
            recent_r = self._ep_rewards[-50:]
            recent_l = self._ep_lengths[-50:]
            recent_s = self._ep_scores[-50:]
            recent_lv = self._ep_levels[-50:]
            
            mean_r = float(np.mean(recent_r)) if recent_r else 0.0
            mean_l = float(np.mean(recent_l)) if recent_l else 0.0
            mean_s = float(np.mean(recent_s)) if recent_s else 0.0
            mean_lv = float(np.mean(recent_lv)) if recent_lv else 1.0
            
            # Draw custom progress bar
            pct = min(100, int((n / self.total_steps) * 100))
            bar_len = 15
            filled = int(bar_len * (pct / 100))
            bar = "█" * filled + "░" * (bar_len - filled)
            
            print(f"\r[{bar}] {pct}% | Steps: {n:>9,} | Elapsed: {elapsed/60:5.1f}m | mean_r: {mean_r:7.2f} | mean_len: {mean_l:5.0f} | mean_score: {mean_s:6.1f} | mean_level: {mean_lv:4.2f} | eps: {len(self._ep_rewards)}", end="", flush=True)
            
            self.mlflow_logger.log_metrics(
                {'mean_reward_50ep': mean_r,
                 'mean_ep_length_50': mean_l,
                 'mean_score_50ep': mean_s,
                 'mean_level_50ep': mean_lv,
                 'elapsed_min': elapsed / 60,
                 'episodes': len(self._ep_rewards)},
                step=n,
            )
            self._next_log = n + self.log_every

        if n >= self._next_ckpt:
            self.model.save(self.checkpoint_path)
            print(f"\n  ✓ checkpoint saved → {self.checkpoint_path}.zip")
            self._next_ckpt = n + self.checkpoint_every

        return True

print(f"Training started — MaskablePPO + CNN, {N_ENVS} parallel envs.\n")
with MLflowLogger(experiment_name='rl_training', run_name='ppo_cnn_masked') as logger:
    logger.log_params({
        'algorithm':         'MaskablePPO',
        'policy':            'CnnPolicy (custom 3xConv)',
        'n_envs':            N_ENVS,
        'n_steps':           N_STEPS,
        'batch_size':        512,
        'learning_rate':     2.5e-4,
        'gamma':             0.99,
        'gae_lambda':        0.95,
        'clip_range':        0.1,
        'ent_coef':          0.01,
        'env':               'PacmanGridEnv',
        'obs_shape':         str(vec_env.observation_space.shape),
        'reward_scale_div':  100.0,
        'pbrs_coef':         0.05,
        'step_penalty':      -0.01,
        'max_steps':         5000,
    })

    callback = ConsoleCallback(
        CHECKPOINT_PATH, CHECKPOINT_EVERY, LOG_EVERY, logger, TOTAL_TIMESTEPS
    )

    try:
        # Determine reset_num_timesteps dynamically based on whether it is a resumed run
        resumed = os.path.exists(CHECKPOINT_PATH + ".zip")
        model.learn(
            total_timesteps=TOTAL_TIMESTEPS,
            callback=callback,
            reset_num_timesteps=not resumed,
            progress_bar=False,
        )
    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Saving final model...")
    finally:
        model.save(CHECKPOINT_PATH)
        logger.log_metrics({'total_timesteps': model.num_timesteps})
        print(f"\nFinal model saved → {CHECKPOINT_PATH}.zip")

Training started — MaskablePPO + CNN, 8 parallel envs.

[░░░░░░░░░░░░░░░] 2% | Steps:   100,000 | Elapsed:   1.6m | mean_r:  -12.73 | mean_len:   200 | mean_score:  425.4 | mean_level: 1.00 | eps: 566
  ✓ checkpoint saved → ..\models\ppo_pacman.zip
[░░░░░░░░░░░░░░░] 4% | Steps:   200,000 | Elapsed:   2.9m | mean_r:  -10.96 | mean_len:   217 | mean_score:  620.6 | mean_level: 1.00 | eps: 1049
  ✓ checkpoint saved → ..\models\ppo_pacman.zip
[░░░░░░░░░░░░░░░] 6% | Steps:   300,000 | Elapsed:   4.4m | mean_r:   -9.55 | mean_len:   224 | mean_score:  768.2 | mean_level: 1.00 | eps: 1493
  ✓ checkpoint saved → ..\models\ppo_pacman.zip
[█░░░░░░░░░░░░░░] 8% | Steps:   400,000 | Elapsed:   5.8m | mean_r:   -8.20 | mean_len:   280 | mean_score:  959.8 | mean_level: 1.00 | eps: 1885
  ✓ checkpoint saved → ..\models\ppo_pacman.zip
[█░░░░░░░░░░░░░░] 10% | Steps:   500,000 | Elapsed:   7.2m | mean_r:   -7.85 | mean_len:   308 | mean_score: 1022.4 | mean_level: 1.00 | eps: 2223
  ✓ checkpoint saved →

## 🎬 Test and Watch the Trained Agent in Action!

Once you believe your agent is trained enough (when its average reward starts climbing and stabilizing), run the cell below to **watch the agent play Pac-Man in real-time** via an ASCII live-render animation!

In [6]:
from IPython.display import clear_output
import time

print("Loading trained model...")
try:
    # Load trained model
    trained_model = MaskablePPO.load(CHECKPOINT_PATH)
    print("Loaded model successfully!")
except Exception as e:
    print(f"Could not load checkpoint: {e}. Standard fallback to current model weights.")
    trained_model = model

# Create the evaluation environment (MUST match PacmanGridEnv to avoid AssertionError)
eval_env = PacmanGridEnv(seed=42)
obs, _ = eval_env.reset()
done = False
total_eval_reward = 0.0
step_num = 0

ROWS, COLS = eval_env._state.maze.shape
ACTION_UP = 0
ACTION_DOWN = 1
ACTION_LEFT = 2
ACTION_RIGHT = 3

while not done:
    # Predict action using the action mask to prevent walking into walls
    action, _ = trained_model.predict(
        obs, 
        action_masks=eval_env.action_masks(), 
        deterministic=True
    )
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_eval_reward += reward
    done = terminated or truncated
    step_num += 1
    
    # Render the board from the environment's internal game state
    clear_output(wait=True)
    state = eval_env._state
    rows = []
    
    # Build a lookup for ghosts at each position
    ghosts_at_pos = {}
    for g in state.ghosts:
        ghosts_at_pos[g.pos] = g

    for r in range(ROWS):
        row_str = ""
        for c in range(COLS):
            pos = (r, c)
            if pos == state.pacman_pos:
                # Clean Unicode direction arrows
                direction = state.pacman_dir
                pac_char = "◀"
                if direction == ACTION_UP:
                    pac_char = "▲"
                elif direction == ACTION_DOWN:
                    pac_char = "▼"
                elif direction == ACTION_LEFT:
                    pac_char = "◀"
                elif direction == ACTION_RIGHT:
                    pac_char = "▶"
                row_str += pac_char
            elif pos in ghosts_at_pos:
                g = ghosts_at_pos[pos]
                if g.eaten:
                    row_str += "E"  # White eaten ghost eyes
                elif state.frightened_timer > 0:
                    row_str += "S"  # Frightened/scared ghost
                else:
                    row_str += g.name[0]  # Signature uppercase letter (B, P, I, C)
            else:
                tile_val = int(state.maze[r, c])
                if tile_val == 1:    # Wall
                    row_str += "█"
                elif tile_val == 2:  # Pellet
                    row_str += "·"
                elif tile_val == 3:  # Power Pellet
                    row_str += "●"
                elif tile_val == 4:  # Door
                    row_str += "═"
                else:
                    row_str += " "
        rows.append(row_str)
    
    print("\n".join(rows))
    print(f"\nSteps: {step_num} | Score: {state.score} | Level: {state.level} | Lives: {state.lives} | Total Reward: {total_eval_reward:.2f}")
    time.sleep(0.08)  # Highly readable visual playback

print("\nGame Over!")


████████████████████████████
█            ██··········· █
█ ████·█████ ██·█████·████ █
█ ████·█████ ██·█████·████ █
█ ████·█████ ██·█████·████ █
█ ··········               █
█ ████·██·████████·██·████ █
█ ████·██·████████·██ ████ █
█      ██····██····██      █
██████ █████ ██ █████ ██████
██████ █████ ██ █████ ██████
██████ ██    B     ██ ██████
██████ ██ ███══███ ██ ██████
██████ ██ █      █ ██ ██████
          █I P  C█          
██████ ██ █      █ ██ ██████
██████ ██ ████████ ██ ██████
██████ ██          ██ ██████
██████ ██ ████████ ██ ██████
██████ ██ ████████ ██ ██████
█            ██            █
█ ████ █████ ██ █████ ████ █
█ ████ █████ ██ █████ ████ █
█   ██       ◀        ██   █
███ ██·██ ████████ ██ ██ ███
███ ██·██ ████████ ██ ██ ███
█   ···██    ██    ██      █
█ ██████████ ██ ██████████ █
█ ██████████ ██ ██████████ █
█                          █
████████████████████████████

Steps: 450 | Score: 2300 | Level: 1 | Lives: 0 | Total Reward: 3.51

Game Over!
